In [ ]:
import ee
import datetime

# Authenticate and initialize Google Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

# USA bounding box
bbox = ee.Geometry.BBox(-171.791110603, 18.91619, -66.96466, 71.3577635769)

# ERA5-Land variables to export
variables = [
    "temperature_2m",
    "skin_temperature",
    "soil_temperature_level_1",
    "soil_temperature_level_2",
    "soil_temperature_level_3",
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
    "volumetric_soil_water_layer_3",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "surface_pressure",
    "total_precipitation_sum",
    "surface_latent_heat_flux_sum",
    "surface_net_solar_radiation_sum",
    "evaporation_from_vegetation_transpiration_sum"
]

start_date = datetime.date(1990, 1, 1)
end_date = datetime.date(2024, 12, 31)

print(f"Exporting ERA5-Land data for USA")
print(f"Date range: {start_date} to {end_date}")
print(f"Bounding box: {bbox.getInfo()['coordinates']}")
print(f"Variables: {len(variables)}")
print(f"{'='*60}\n")

current_date = start_date
task_count = 0

while current_date <= end_date:
    next_date = current_date + datetime.timedelta(days=1)
    date_str = current_date.strftime("%Y-%m-%d")
    next_date_str = next_date.strftime("%Y-%m-%d")

    dataset = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
               .filterBounds(bbox)
               .filterDate(date_str, next_date_str)
               .select(variables))

    image = dataset.mean().clip(bbox)

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f"ERA5_USA_{date_str}",
        folder="USA_ERA5_Daily",
        fileNamePrefix=f"ERA5_USA_{date_str}",
        region=bbox,
        scale=10000,
        crs='EPSG:4326',
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )

    task.start()
    task_count += 1
    
    if task_count % 30 == 0:  # Print progress every 30 days
        print(f"Exported {task_count} tasks. Latest: {date_str}")

    current_date = next_date

print(f"\n{'='*60}")
print(f"All {task_count} daily export tasks initiated!")
print(f"Monitor tasks at: https://code.earthengine.google.com/tasks")
print(f"{'='*60}")